# `_common` — shared configuration and helpers
Included by every notebook with `%run ./_common`. Holds widget parameters, table metadata,
data-quality rules and the Delta helpers used by the Bronze / Silver / Gold notebooks.

In [ ]:
from datetime import date

from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F


def _widget(name, default=""):
    try:
        dbutils.widgets.text(name, default)
    except Exception:
        pass
    return dbutils.widgets.get(name)


CATALOG = _widget("catalog", "tradefin")
LANDING_ROOT = _widget("landing_root", "").rstrip("/")      # abfss://landing@<account>.dfs.core.windows.net
LAKEHOUSE_ROOT = _widget("lakehouse_root", "").rstrip("/")  # abfss://lakehouse@<account>.dfs.core.windows.net
AS_OF = _widget("as_of_date", "2025-06-30")                 # business date of the Gold snapshot
SECRET_SCOPE = _widget("secret_scope", "tradefin")          # Key Vault-backed scope (source reconciliation)


def fq(schema: str, table: str) -> str:
    """Fully qualified, lower-case table name."""
    return f"{CATALOG}.{schema}.{table.lower()}"


def sk(*cols):
    """Deterministic surrogate key: xxhash64 over the string form of the business key columns."""
    return F.xxhash64(*[(F.col(c) if isinstance(c, str) else c).cast("string") for c in cols])

In [ ]:
# Source table metadata. kind: F = full refresh, I = watermark incremental (upsert), A = append-only.
TABLES = {
    "BANK": dict(kind="F", pk=["bank_id"], order=None,
                 cols=["bank_id", "bic", "bank_name", "country"]),
    "CURRENCY_FX": dict(kind="F", pk=["currency_code"], order=None,
                        cols=["currency_code", "currency_name", "rate_to_usd", "rate_date"]),
    "PARTY": dict(kind="I", pk=["party_id"], order="updated_at",
                  cols=["party_id", "party_name", "country", "party_type", "risk_rating", "created_at", "updated_at"]),
    "LETTER_OF_CREDIT": dict(kind="I", pk=["lc_id"], order="updated_at",
                             cols=["lc_id", "lc_number", "issuing_bank_id", "amount", "currency", "issue_date",
                                   "expiry_date", "payment_terms", "status", "created_at", "updated_at"]),
    "LC_PARTY_ROLE": dict(kind="I", pk=["lc_id", "party_id", "role"], order="updated_at",
                          cols=["lc_id", "party_id", "role", "valid_from", "valid_to", "updated_at"]),
    "LC_AMENDMENT": dict(kind="I", pk=["amendment_id"], order="updated_at",
                         cols=["amendment_id", "lc_id", "version_no", "effective_at", "amount_delta",
                               "expiry_change", "approval_status", "updated_at"]),
    "LC_EXPOSURE_EVENT": dict(kind="A", pk=["event_id"], order=None,
                              cols=["event_id", "lc_id", "exposure_amount", "currency", "exposure_type",
                                    "lifecycle_status", "occurred_at", "source_event_ref"]),
}
# Column used as source_updated_at in Bronze
SOURCE_UPDATED_COL = {"PARTY": "updated_at", "LETTER_OF_CREDIT": "updated_at", "LC_PARTY_ROLE": "updated_at",
                      "LC_AMENDMENT": "updated_at", "LC_EXPOSURE_EVENT": "occurred_at"}
OPERATION = {"F": "SNAPSHOT", "I": "UPSERT", "A": "INSERT"}

# Referential checks applied in Silver: (column in this table, parent silver table, parent column)
ORPHAN_CHECKS = {
    "LC_PARTY_ROLE": [("lc_id", "LETTER_OF_CREDIT", "lc_id"), ("party_id", "PARTY", "party_id")],
    "LC_AMENDMENT": [("lc_id", "LETTER_OF_CREDIT", "lc_id")],
    "LC_EXPOSURE_EVENT": [("lc_id", "LETTER_OF_CREDIT", "lc_id")],
}

RATINGS = ["AAA", "AA+", "AA", "AA-", "A+", "A", "A-", "BBB+", "BBB", "BBB-", "BB+", "BB", "B", "CCC", "CC", "C", "D", "NR"]


def _in(values):
    return ", ".join(f"'{v}'" for v in values)


def get_rules(table: str):
    """(rule_name, SQL predicate that must be TRUE for a valid row). NULL counts as a failure."""
    def currencies():
        return [r[0] for r in spark.table(fq("silver", "CURRENCY_FX")).select("currency_code").collect()]

    if table == "BANK":
        return [("bank_id_not_null", "bank_id IS NOT NULL"), ("bic_not_null", "bic IS NOT NULL")]
    if table == "CURRENCY_FX":
        return [("currency_code_len3", "length(currency_code) = 3"), ("rate_positive", "rate_to_usd > 0")]
    if table == "PARTY":
        return [("party_id_not_null", "party_id IS NOT NULL"),
                ("party_name_present", "party_name IS NOT NULL AND trim(party_name) <> ''"),
                ("risk_rating_valid", f"risk_rating IN ({_in(RATINGS)})"),
                ("country_len2", "length(country) = 2")]
    if table == "LETTER_OF_CREDIT":
        return [("lc_id_not_null", "lc_id IS NOT NULL"), ("amount_positive", "amount > 0"),
                ("currency_known", f"currency IN ({_in(currencies())})"),
                ("expiry_not_before_issue", "expiry_date >= issue_date"),
                ("status_valid", "status IN ('ISSUED','AMENDED','PRESENTED','SETTLED','EXPIRED')")]
    if table == "LC_PARTY_ROLE":
        return [("role_valid", "role IN ('APPLICANT','BENEFICIARY')"), ("valid_from_not_null", "valid_from IS NOT NULL")]
    if table == "LC_AMENDMENT":
        return [("lc_id_not_null", "lc_id IS NOT NULL"),
                ("approval_status_valid", "approval_status IN ('APPROVED','PENDING','REJECTED')")]
    if table == "LC_EXPOSURE_EVENT":
        return [("lc_id_not_null", "lc_id IS NOT NULL"), ("exposure_non_negative", "exposure_amount >= 0"),
                ("currency_known", f"currency IN ({_in(currencies())})"),
                ("exposure_type_valid", "exposure_type IN ('CONTINGENT','FUNDED')"),
                ("source_event_ref_present", "source_event_ref IS NOT NULL")]
    raise ValueError(table)


def split_valid(df, rules):
    """Return (valid_rows, invalid_rows); invalid rows carry the array column `_failed`."""
    checks = [F.when(~F.coalesce(F.expr(cond), F.lit(False)), F.lit(name)) for name, cond in rules]
    flagged = df.withColumn("_failed", F.filter(F.array(*checks), lambda x: x.isNotNull()))
    return flagged.where(F.size("_failed") == 0).drop("_failed"), flagged.where(F.size("_failed") > 0)

In [ ]:
def upsert(df, target: str, keys, order_col=None):
    """Idempotent Delta MERGE. With order_col, a stale (older) source row never overwrites a newer target row."""
    if not spark.catalog.tableExists(target):
        df.write.format("delta").saveAsTable(target)
        return
    cond = " AND ".join(f"t.{k} <=> s.{k}" for k in keys)
    merge = DeltaTable.forName(spark, target).alias("t").merge(df.alias("s"), cond)
    if order_col:
        merge = merge.whenMatchedUpdateAll(condition=f"s.{order_col} >= t.{order_col}")
    else:
        merge = merge.whenMatchedUpdateAll()
    merge.whenNotMatchedInsertAll().execute()


def insert_only(df, target: str, keys):
    """Append rows whose key is not already in the target (replay-safe)."""
    if not spark.catalog.tableExists(target):
        df.write.format("delta").saveAsTable(target)
        return
    cond = " AND ".join(f"t.{k} <=> s.{k}" for k in keys)
    DeltaTable.forName(spark, target).alias("t").merge(df.alias("s"), cond).whenNotMatchedInsertAll().execute()


def quarantine(bad_df, table: str):
    """Append invalid rows (with the rules they failed) to silver.quarantine."""
    pk = TABLES[table]["pk"]
    payload = [c for c in bad_df.columns if c != "_failed"]
    out = bad_df.select(
        F.lit(table).alias("source_table"),
        F.col("batch_id").cast("string").alias("batch_id"),
        F.concat_ws("|", *[F.col(k).cast("string") for k in pk]).alias("record_key"),
        F.to_json(F.struct(*[F.col(c) for c in payload])).alias("record_json"),
        F.col("_failed").alias("failed_rules"),
        F.current_timestamp().alias("quarantined_at"),
    )
    out.write.format("delta").mode("append").saveAsTable(fq("silver", "quarantine"))